## Notebook 1: Data Exploration and Cleaning
 
##  Objective
The main goal of this notebook is to load the raw books dataset, discover any inconsistencies (missing values, duplicates, bad text), and clean it. By the end of this notebook, we will have a perfectly clean dataset ready for Natural Language Processing (NLP) and Deep Learning.

##  Dataset Info
* **Source:** `Books_dataset.csv`
* **Features:** title, target_genre, text_sample 
* **Target Variable:** `target_genre` (21 unique genres).

---
##  Step 1: Data Loading & Initial Inspection


In [2]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
df = pd.read_csv("Books_dataset.csv", encoding='latin-1', usecols=['title', 'target_genre', 'text_sample'])
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("-" * 40)
display(df.head(3))

Shape: 6977 rows, 3 columns
----------------------------------------


,title,target_genre,text_sample
0,2001,Science Fiction,A novel that proposes an idea about how the hu...
1,De la terre Ã la lune,Science Fiction,Novela grafica Mientras la guerra federal asol...
2,The book of the damned,Science Fiction,The Book of the Damned was the first published...


---
##  Step 2: Data Quality Assessment (Finding Errors)


In [3]:
print("1. Checking for Missing Values (Nulls) ")
print(df.isnull().sum())
print("\n" + "="*40 + "\n")

print("2. Checking for Duplicates ")
duplicates = df.duplicated().sum()
print(f"Total duplicate rows found: {duplicates}")
print("\n" + "="*40 + "\n")

print("3. Raw Word Count Statistics ")
raw_word_counts = df['text_sample'].apply(lambda x: len(str(x).split()))
print(raw_word_counts.describe())

short_texts = (raw_word_counts < 10).sum()
if short_texts > 0:
    print(f"\n[Warning]: Found {short_texts} books with very short descriptions (< 10 words).")

1. Checking for Missing Values (Nulls) 
title           3
target_genre    3
text_sample     4
dtype: int64


2. Checking for Duplicates 
Total duplicate rows found: 2


3. Raw Word Count Statistics 
count    6977.000000
mean      159.900244
std       107.611762
min         1.000000
25%       102.000000
50%       135.000000
75%       189.000000
max      5229.000000
Name: text_sample, dtype: float64

[Warning]: Found 7 books with very short descriptions (< 10 words).


---
##  Step 3: Data Cleaning & Preprocessing
Based on our assessment, we will apply the following fixes:
- Strip all non-alphabetic characters (numbers, punctuation).
- Convert text to lowercase.
- Remove English stopwords.
- Apply WordNet Lemmatization to reduce words to their base form.

In [4]:
df.dropna(subset=['title', 'target_genre', 'text_sample'], inplace=True)
df.drop_duplicates(inplace=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text_pipeline(text):
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.lower().split()
    
    clean_words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) > 2]
    return " ".join(clean_words)

print("THE DATASET IS BEING CLEANED")

df['cleaned_text'] = df['text_sample'].apply(clean_text_pipeline)

df['word_count'] = df['cleaned_text'].apply(lambda x: len(str(x).split()))
df_final = df[df['word_count'] >= 10].copy()

df_final = df_final[['title', 'target_genre', 'cleaned_text']]
df_final.to_csv("BooksClassifier_dataset_cleaned.csv", index=False, encoding="utf-8")

print(f"The cleaned dataset is ready ({len(df_final)} rows) ==> 'BooksClassifier_dataset_cleaned.csv'")

THE DATASET IS BEING CLEANED
The cleaned dataset is ready (6967 rows) ==> 'BooksClassifier_dataset_cleaned.csv'


---
##  Step 4: Data Quality Recheck (Finding Errors In Cleaned Data)


In [5]:
import pandas as pd
print("Loading the final saved dataset to verify\n")
df_final_check = pd.read_csv("BooksClassifier_dataset_cleaned.csv")

print("1. Final Shape & Null Check")
print(f"Total Rows: {len(df_final_check)}")
print("Null Values:")
print(df_final_check.isnull().sum())
print("\n" + "="*40 + "\n")

print("2. Final Word Count Stats ")
word_counts = df_final_check['cleaned_text'].apply(lambda x: len(str(x).split()))
print(word_counts.describe())

min_words = word_counts.min()
print(f"\nMinimum words in a text: {min_words} (Should be >= 10)")
print("\n" + "="*40 + "\n")

print("3. Random Sample Check")
sample = df_final_check.sample(1)
print(f"Title: {sample['title'].values[0]}")
print(f"Genre: {sample['target_genre'].values[0]}")
print(f"Cleaned Text (First 300 chars):\n{str(sample['cleaned_text'].values[0])[:300]}...")

Loading the final saved dataset to verify

1. Final Shape & Null Check
Total Rows: 6967
Null Values:
title           0
target_genre    0
cleaned_text    0
dtype: int64


2. Final Word Count Stats 
count    6967.000000
mean       88.697287
std        60.442626
min        25.000000
25%        56.000000
50%        75.000000
75%       105.000000
max      2972.000000
Name: cleaned_text, dtype: float64

Minimum words in a text: 25 (Should be >= 10)


3. Random Sample Check
Title: Murder in Victorian Scotland
Genre: True Crime
Cleaned Text (First 300 chars):
new look life trial madeleine smith young scottish woman accused poisoning undesired suitor book us analysis smith correspondence victim trial testimony reveal much victorian society scottish law woman received nebulous verdict proven verdict proven unique scotland allowing defendant free verdict of...
